# Светлана: обучение LoRA на Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/paulafanasyev/Svetlana-2.0/blob/training-master-dataset-2026-10/training/gemma4/svetlana_colab.ipynb)

Среда: **Runtime → Change runtime type → T4 GPU** (бесплатный Colab подходит). Запускайте ячейки по порядку.

Что делает блокнот: клонирует репозиторий, собирает и проверяет датасет (схема, дубли, утечка eval, sha256), меряет базовую модель на отложенном eval, обучает LoRA, меряет снова и сохраняет адаптер на Google Drive.

Честные статусы: снижение loss не доказывает качество. Итог `EVAL_PASSED` означает только, что пройден keyword-eval. Для продакшена нужна ручная проверка ответов (статус остаётся NOT PROVEN до неё).

In [ ]:
# 1. Настройки
BRANCH = "training-master-dataset-2026-10"
BASE_MODEL = "unsloth/gemma-4-E2B-it"          # основная модель
FALLBACK_MODELS = ["google/gemma-4-E2B-it", "unsloth/gemma-3-4b-it"]  # если основная не загрузилась
MAX_SEQ = 1536
EPOCHS = 2
LR = 1e-4
LORA_R = 16
SAVE_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/svetlana_lora"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Установка (3-5 минут)
!pip -q install -U unsloth unsloth_zoo
!pip -q install -U "transformers" "trl" "datasets"

In [ ]:
# 3. Код и данные
import os
if not os.path.exists("/content/Svetlana-2.0"):
    !git clone --depth 1 -b {BRANCH} https://github.com/paulafanasyev/Svetlana-2.0.git /content/Svetlana-2.0
%cd /content/Svetlana-2.0
!python training/datasets/marketplace/build_marketplace_dataset.py

In [ ]:
# 4. Сборка и проверка датасета
import json, hashlib, glob
EXPECTED_SHA = {
    "training/datasets/marketplace/svetlana_marketplace_train_v1.jsonl": "ee6b56e5d3f6cd71a373ea32008dd3f5c232e6de6e7686962b0bae29c295878f",
    "training/evaluation/svetlana_marketplace_eval_v1.jsonl": "62fd88c7490e19f7da0041fe227abfc9d9723ab61503b54c2b9200883a13111d",
}
TRAIN_FILES = [
    "training/datasets/svetlana_master_training_v1.jsonl",
    "training/datasets/marketplace/svetlana_marketplace_train_v1.jsonl",
    "training/datasets/svetlana_seed.jsonl",
    "training/datasets/svetlana_capability_seed.jsonl",
    "training/datasets/svetlana_capability_training_v2.jsonl",
    "training/datasets/svetlana_advanced_training_v1.jsonl",
    "training/datasets/self_employed_official.jsonl",
]
EVAL_FILES = ["training/evaluation/svetlana_marketplace_eval_v1.jsonl"]
def sha(p): return hashlib.sha256(open(p, "rb").read()).hexdigest()
for p, h in EXPECTED_SHA.items():
    got = sha(p); print(("OK " if got == h else "MISMATCH ") + p, got[:12])
    if got != h: print("  ВНИМАНИЕ: хэш отличается от записанного в README, датасет пересобран другой версией генератора")
def text_of(c):
    return c if isinstance(c, str) else " ".join(x.get("text", "") for x in c)
def valid(rec):
    m = rec.get("messages")
    if not isinstance(m, list) or len(m) < 2: return False
    roles = [x.get("role") for x in m]
    if m[-1]["role"] != "assistant": return False
    if any(r not in ("system", "user", "assistant") for r in roles): return False
    return all(text_of(x.get("content", "")).strip() for x in m)
train, seen, stats = [], set(), {}
for p in TRAIN_FILES:
    if not os.path.exists(p):
        print("нет файла, пропуск:", p); continue
    ok = bad = dup = 0
    for line in open(p, encoding="utf-8"):
        if not line.strip(): continue
        try: rec = json.loads(line)
        except json.JSONDecodeError: bad += 1; continue
        if not valid(rec): bad += 1; continue
        msgs = [{"role": x["role"], "content": text_of(x["content"])} for x in rec["messages"]]
        k = hashlib.sha1(json.dumps(msgs, ensure_ascii=False).encode()).hexdigest()
        if k in seen: dup += 1; continue
        seen.add(k); train.append(msgs); ok += 1
    stats[p] = (ok, bad, dup)
    print(f"{p}: ok={ok} bad={bad} dup={dup}")
eval_cases = [json.loads(l) for f in EVAL_FILES for l in open(f, encoding="utf-8") if l.strip()]
train_users = {m["content"] for t in train for m in t if m["role"] == "user"}
leak = [c["id"] for c in eval_cases if c["messages"][-1]["content"] in train_users and c["messages"][-1]["role"] == "user" and not c["messages"][-1]["content"].startswith("<tool_result>")]
print("eval кейсов:", len(eval_cases), "утечек:", len(leak))
assert not leak, f"eval пересекается с train: {leak}"
DATASET_SHA = hashlib.sha256("\n".join(json.dumps(t, ensure_ascii=False) for t in train).encode()).hexdigest()
print("train примеров:", len(train), "sha256 итогового набора:", DATASET_SHA)

In [ ]:
# 5. Модель
from unsloth import FastLanguageModel
import torch
model = tok = None
for name in [BASE_MODEL] + FALLBACK_MODELS:
    try:
        model, tok = FastLanguageModel.from_pretrained(model_name=name, max_seq_length=MAX_SEQ, load_in_4bit=True)
        MODEL_USED = name; print("загружена", name); break
    except Exception as e:
        print("не загрузилась", name, "->", repr(e)[:200])
assert model is not None
text_tok = getattr(tok, "tokenizer", tok)
def render(msgs, add_gen=False):
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=add_gen)
    except Exception:
        mm = [{"role": m["role"], "content": [{"type": "text", "text": m["content"]}]} for m in msgs]
        return tok.apply_chat_template(mm, tokenize=False, add_generation_prompt=add_gen)
print(render(train[0])[:600])

In [ ]:
# 6. Отложенный eval (keyword-рубрика)
def generate(msgs, max_new=256):
    prompt = render(msgs, add_gen=True)
    ids = text_tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new, do_sample=False)
    return text_tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
def run_eval(tag):
    FastLanguageModel.for_inference(model)
    rows, passed = [], 0
    for c in eval_cases:
        ans = generate(c["messages"])
        low = ans.lower()
        ok_any = any(k.lower() in low for k in c["expect"]["any"]) if c["expect"]["any"] else True
        ok_none = not any(k.lower() in low for k in c["expect"]["none"])
        ok = ok_any and ok_none; passed += ok
        rows.append({"id": c["id"], "category": c["category"], "pass": ok, "answer": ans[:400]})
    rate = passed / len(eval_cases)
    print(f"{tag}: {passed}/{len(eval_cases)} = {rate:.0%}")
    return rate, rows
base_rate, base_rows = run_eval("база")

In [ ]:
# 7. Обучение LoRA
import inspect
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, lora_alpha=LORA_R, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=3407)
texts = [render(t) for t in train]
lens = [len(text_tok(x, add_special_tokens=False)["input_ids"]) for x in texts]
texts = [x for x, n in zip(texts, lens) if n <= MAX_SEQ]
print("в обучение:", len(texts), "отброшено длинных:", len(lens) - len(texts))
ds = Dataset.from_list([{"text": x} for x in texts]).shuffle(seed=3407)
cfg = dict(dataset_text_field="text", per_device_train_batch_size=1, gradient_accumulation_steps=4,
           num_train_epochs=EPOCHS, learning_rate=LR, warmup_steps=10, lr_scheduler_type="cosine",
           logging_steps=10, optim="adamw_8bit", weight_decay=0.01, seed=3407, output_dir="outputs",
           report_to="none", save_strategy="no")
params = inspect.signature(SFTConfig).parameters
cfg["max_length" if "max_length" in params else "max_seq_length"] = MAX_SEQ
if "dataset_kwargs" in params: cfg["dataset_kwargs"] = {"add_special_tokens": False}
kw = {"processing_class": tok} if "processing_class" in inspect.signature(SFTTrainer).parameters else {"tokenizer": tok}
trainer = SFTTrainer(model=model, train_dataset=ds, args=SFTConfig(**cfg), **kw)
# loss только на ответах ассистента (если узнаём разметку шаблона)
try:
    from unsloth.chat_templates import train_on_responses_only
    if "<start_of_turn>model\n" in texts[0]:
        trainer = train_on_responses_only(trainer, instruction_part="<start_of_turn>user\n", response_part="<start_of_turn>model\n")
        print("loss: только ответы ассистента")
    else:
        print("ВНИМАНИЕ: разметка шаблона не распознана, loss по всей последовательности")
except Exception as e:
    print("train_on_responses_only недоступен:", e)
stats_train = trainer.train()
print(stats_train)

In [ ]:
# 8. Повторный eval и отчёт
tuned_rate, tuned_rows = run_eval("после LoRA")
status = "EVAL_PASSED" if tuned_rate >= 0.9 and tuned_rate >= base_rate else "EVAL_FAILED"
report = {"model": MODEL_USED, "dataset_sha256": DATASET_SHA, "train_examples": len(texts),
          "baseline_pass_rate": base_rate, "tuned_pass_rate": tuned_rate, "status": status,
          "production": "NOT PROVEN: нужна ручная проверка ответов", "rows": tuned_rows}
json.dump(report, open("svetlana_eval_report.json", "w"), ensure_ascii=False, indent=1)
for r in tuned_rows:
    if not r["pass"]: print("FAIL", r["id"], r["category"], "|", r["answer"][:200])
print(status)

In [ ]:
# 9. Сохранение адаптера
import shutil
out = "svetlana_lora_adapter"
model.save_pretrained(out); tok.save_pretrained(out)
shutil.copy("svetlana_eval_report.json", out)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    dst = os.path.join(DRIVE_DIR, DATASET_SHA[:12])
    shutil.copytree(out, dst, dirs_exist_ok=True)
    print("сохранено:", dst)
# GGUF для llama.cpp/Ollama (долго, по желанию):
# model.save_pretrained_gguf("svetlana_gguf", tok, quantization_method="q4_k_m")

In [ ]:
# 10. Поговорить со Светланой
SYS = train[0][0]["content"] if train[0][0]["role"] == "system" else ""
q = "Отмени мой заказ СВ-1234"
print(generate([{"role": "system", "content": SYS}, {"role": "user", "content": q}]))